# Drive Wise - Metadata Aware Automotive RAG Assistant

Welcome to the **Drive Wise** demo notebook. This notebook provides a standalone walkthrough of the retrieval and generation pipeline used in the Drive Wise RAG system. It is designed to allow mentors or developers to test the grounded question-answering capabilities using the official brochure indices.

### Core Pipeline Features:
1. **Metadata Pre-Filtering**: Automatically filters segments to match ONLY the selected car brand and model (e.g., Honda Amaze, Tata Sierra).
2. **Vector Similarity Search**: Embeds queries and ranks matching paragraphs using cosine similarity.
3. **Grounded Generation**: Prompts `gemini-3.5-flash` to write answers strictly based on the retrieved brochure facts, with inline source citations.

---

### 1. Installation & Settings
Ensure you have the required packages installed. You will also need to configure your Google Gemini API key.

In [ ]:
# Install dependencies if not already installed
!pip install google-generativeai pypdf numpy

In [ ]:
import os
import pickle
import numpy as np
import google.generativeai as genai
from google.colab import userdata # Use Google Colab Secrets or standard env var

# Configure API Key
try:
    # If running on Colab, load from secret storage
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    # Otherwise load from environment variable
    api_key = os.environ.get('GOOGLE_API_KEY', 'YOUR_GEMINI_API_KEY_HERE')

genai.configure(api_key=api_key)
print("Gemini API configured successfully.")

### 2. Loading the Brochure Index
Load the pre-computed index that contains the parsed brochure chunks, metadata (page, section), and embeddings.

In [ ]:
INDEX_PATH = os.path.join("index", "brochure_index.pkl")

if not os.path.exists(INDEX_PATH):
    print(f"Index not found at {INDEX_PATH}. Please make sure you run 'indexer.py' first or upload the index folder.")
else:
    with open(INDEX_PATH, 'rb') as f:
        index_data = pickle.load(f)
    
    # List available cars
    files = index_data.get("files", {})
    print("Indexed Car Brochures:")
    for filename, meta in files.items():
        print(f"- {meta.get('brand')} {meta.get('model')} (Chunks: {meta.get('chunks_count')})")

### 3. Defining the Retrieval Engine
Define metadata-filtering, cosine similarity calculations, and top-K chunk retrieval.

In [ ]:
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

def calculate_keyword_score(query, text):
    query_words = [w.strip("?,.:;!\"'()").lower() for w in query.split()]
    if not query_words:
        return 0.0
    text_lower = text.lower()
    matches = sum(1 for w in query_words if w in text_lower)
    return matches / len(query_words)

def retrieve_chunks(query, brand, model, limit=4):
    chunks = index_data.get("chunks", [])
    
    # 1. Metadata Pre-Filtering
    filtered = [c for c in chunks if c.get("brand", "").lower() == brand.lower() and c.get("model", "").lower() == model.lower()]
    if not filtered:
        print(f"No brochure text found for {brand} {model}")
        return []
        
    # 2. Embed user query
    emb_resp = genai.embed_content(model='models/gemini-embedding-001', content=query)
    query_embedding = emb_resp['embedding']
    
    # 3. Score chunks
    scored = []
    for chunk in filtered:
        sem_score = cosine_similarity(query_embedding, chunk['embedding'])
        key_score = calculate_keyword_score(query, chunk['text'])
        hybrid_score = 0.8 * sem_score + 0.2 * key_score
        
        scored.append({
            "text": chunk["text"],
            "section": chunk["section"],
            "page": chunk["page"],
            "source_file": chunk["source_file"],
            "score": hybrid_score
        })
        
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:limit]

### 4. Grounded Response Generation
Prompt Gemini 3.5 Flash to write answers based strictly on retrieved chunks.

In [ ]:
def ask_drivewise(query, brand, model):
    # Retrieve chunks
    chunks = retrieve_chunks(query, brand, model, limit=4)
    if not chunks:
        return "Error: Car brand/model details not found in index.", []
        
    # Format Context
    context_str = ""
    for idx, c in enumerate(chunks):
        context_str += f"\n--- Source [{idx+1}] (Page {c['page']}, Section: {c['section']}) ---\n{c['text']}\n"
        
    system_instruction = f"""
    You are an expert automotive assistant. Answer the user's query about the car: {brand} {model}.
    
    Rules:
    1. Ground your answer ONLY in the provided brochure excerpts under the "Brochure Context" section.
    2. If the info is not in the brochure context, state: "I'm sorry, but that information is not available in the brochure details."
    3. Cite sources inline using numbers like [1], [2], etc., matching the context indexes.
    """
    
    prompt = f"""
    Brochure Context for {brand} {model}:
    {context_str}
    
    User Query: \"{query}\"
    
    Grounded Answer:
    """
    
    model_gen = genai.GenerativeModel(
        model_name="models/gemini-2.5-flash",
        system_instruction=system_instruction
    )
    
    response = model_gen.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.1)
    )
    
    return response.text.strip(), chunks

### 5. Interactive Testing
Ask queries about the indexed models.

In [ ]:
# Example 1: Honda Amaze Safety
query = "What standard safety features does the Honda Amaze have?"
answer, sources = ask_drivewise(query, "Honda", "Amaze")

print("=== ANSWER ===")
print(answer)
print("\n=== RETRIEVED SOURCES ===")
for idx, s in enumerate(sources):
    print(f"[{idx+1}] File: {s['source_file']} | Page: {s['page']} | Section: {s['section']} (Relevance Score: {s['score']:.4f})")

In [ ]:
# Example 2: Tata Sierra Tech features
query = "What is the HypAR HUD feature of the Tata Sierra?"
answer, sources = ask_drivewise(query, "Tata", "Sierra")

print("=== ANSWER ===")
print(answer)
print("\n=== RETRIEVED SOURCES ===")
for idx, s in enumerate(sources):
    print(f"[{idx+1}] File: {s['source_file']} | Page: {s['page']} | Section: {s['section']} (Relevance Score: {s['score']:.4f})")